In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, TensorDataset, DataLoader
from tab_transformer_pytorch import TabTransformer, FTTransformer
from preprocessing import get_features_and_target
from sklearn.preprocessing import LabelEncoder
from RMSELoss import RMSELoss
import plotly.graph_objects as go
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion
from sklearn.model_selection import train_test_split

In [2]:
import huggingface_hub
huggingface_hub.login()

# Getting Dataframe

In [3]:
# Load the training and development datasets
train_df = pd.read_csv("data/train_data.csv")
dev_df = pd.read_csv("data/development_data.csv")

target_column = "PullTest (N)"  

x_train, y_train = get_features_and_target(train_df, target_column)
x_dev, y_dev = get_features_and_target(dev_df, target_column)

# Add Physical Column

In [4]:
def compute_physical_calc(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     f_pull = (np.pi/4) * (4 * np.sqrt(t))**2 * (0.7 * 365) 
     return np.round(f_pull, 1) 

x_train['Physical_Calc'] = compute_physical_calc(x_train) 
x_dev['Physical_Calc'] = compute_physical_calc(x_dev)

In [5]:
x_train.head()

,Pressure (PSI),Welding Time (ms),Angle (Deg),Force (N),Current (A),Thickness A (mm),Thickness B (mm),Physical_Calc
0,35,200,0,6.82,1081.47,0.922,0.920,2953.9
1,35,1500,0,52.25,2014.73,0.920,0.925,2953.9
2,95,200,0,16.57,1321.93,0.912,0.924,2928.2
3,95,200,0,41.42,1615.83,0.948,0.939,3014.9
4,35,1500,0,63.82,1137.29,0.930,0.937,2986.0


In [6]:
y_train.head()

0    2127.7
1    5346.4
2    2350.4
3    2174.8
4    3897.5
Name: PullTest (N), dtype: float64

# Fit Model

In [7]:
# Initialize the regressor
regressor = TabPFNRegressor()  # Uses TabPFN-2.5 weights, trained on synthetic data only.
# To use TabPFN v2:
# regressor = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
regressor.fit(x_train, y_train)

# Predict on the test set
predictions = regressor.predict(x_dev)

# Check Validation Data

In [8]:
import plotly.graph_objects as go
import numpy as np

# Convert to numpy arrays
true_vals = np.array(y_dev).ravel()
pred_vals = np.array(predictions).ravel()

# Sample index
sample_idx = np.arange(len(true_vals))

# Category array (must be aligned with y_dev)
categories = dev_df.groupby("Sample ID")["Category"].first().values

# Masks for each category
mask_good    = categories == "Good"
mask_bad     = categories == "Bad"
mask_explode = categories == "Explode"

fig = go.Figure()

# --- GOOD (circles) ---
fig.add_trace(go.Scatter(
    x=sample_idx[mask_good],
    y=true_vals[mask_good],
    mode="markers",
    name="Good (True)",
    marker=dict(symbol="circle", color="red", size=7)
))

fig.add_trace(go.Scatter(
    x=sample_idx[mask_good],
    y=pred_vals[mask_good],
    mode="markers",
    name="Good (Pred)",
    marker=dict(symbol="circle", color="blue", size=7)
))

# --- BAD (X) ---
fig.add_trace(go.Scatter(
    x=sample_idx[mask_bad],
    y=true_vals[mask_bad],
    mode="markers",
    name="Bad (True)",
    marker=dict(symbol="x", color="red", size=9)
))

fig.add_trace(go.Scatter(
    x=sample_idx[mask_bad],
    y=pred_vals[mask_bad],
    mode="markers",
    name="Bad (Pred)",
    marker=dict(symbol="x", color="blue", size=9)
))

# --- EXPLODE (triangle-up) ---
fig.add_trace(go.Scatter(
    x=sample_idx[mask_explode],
    y=true_vals[mask_explode],
    mode="markers",
    name="Explode (True)",
    marker=dict(symbol="triangle-up", color="red", size=9)
))

fig.add_trace(go.Scatter(
    x=sample_idx[mask_explode],
    y=pred_vals[mask_explode],
    mode="markers",
    name="Explode (Pred)",
    marker=dict(symbol="triangle-up", color="blue", size=9)
))

# Optional: connecting lines for each sample
for i in range(len(sample_idx)):
    fig.add_trace(go.Scatter(
        x=[sample_idx[i], sample_idx[i]],
        y=[true_vals[i], pred_vals[i]],
        mode="lines",
        line=dict(color="gray", width=1),
        showlegend=False
    ))

fig.update_layout(
    title="Validation Samples: True vs Prediction (TabPFN) by Category",
    xaxis_title="Sample Index",
    yaxis_title="Pull Force",
    template="seaborn"
)

fig.show()


# Check Validation Loss and R2

In [9]:
# Calculate MAE and RMSE and R2
mae  = mean_absolute_error(y_dev, predictions)
rmse = np.sqrt(mean_squared_error(y_dev, predictions))
R2   = r2_score(y_dev, predictions)


print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {R2:.2f}")


MAE:  128.01
RMSE: 222.43
R2: 0.61
